In [1]:
import pandas as pd
import re
import warnings
from sklearn.model_selection import train_test_split
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score,precision_score, f1_score, confusion_matrix, mean_absolute_error
from typing import List
import json

pd.set_option('display.width', 200)
pd.set_option('display.max.columns', 200)
pd.set_option('display.max.rows', 50)
warnings.filterwarnings('ignore')

In [2]:
dtype = {
    'SaleId': 'str',
    'name': 'int',
    'regDate': 'str',
    'creatDate': 'str',
    'regionCode': 'str',
    'offerType': 'int'
}
df = pd.read_csv(r'E:\received files\WeChat Files\xwechat_files\wxid_9s3rh9hmnb3822_a189\msg\file\2025-10\used_car_train_20200313.csv', sep=' ', dtype=dtype)
test_df_a = pd.read_csv(r'E:\received files\WeChat Files\xwechat_files\wxid_9s3rh9hmnb3822_a189\msg\file\2025-10\used_car_testA_20200313.csv', sep=' ', dtype=dtype)
test_df_b = pd.read_csv(r'E:\received files\WeChat Files\xwechat_files\wxid_9s3rh9hmnb3822_a189\msg\file\2025-10\used_car_testB_20200421.csv', sep=' ', dtype=dtype)
test_df = pd.concat([test_df_a, test_df_b], axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 31 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   SaleID             150000 non-null  int64  
 1   name               150000 non-null  int32  
 2   regDate            150000 non-null  object 
 3   model              149999 non-null  float64
 4   brand              150000 non-null  int64  
 5   bodyType           145494 non-null  float64
 6   fuelType           141320 non-null  float64
 7   gearbox            144019 non-null  float64
 8   power              150000 non-null  int64  
 9   kilometer          150000 non-null  float64
 10  notRepairedDamage  150000 non-null  object 
 11  regionCode         150000 non-null  object 
 12  seller             150000 non-null  int64  
 13  offerType          150000 non-null  int32  
 14  creatDate          150000 non-null  object 
 15  price              150000 non-null  int64  
 16  v_

In [3]:
df1 = df

In [4]:
df['regYear'] = df['regDate'].apply(lambda x: x[:4])
df['regMonth'] = df['regDate'].apply(lambda x: x[4:6])
df['regDay'] = df['regDate'].apply(lambda x: x[6:])
df = df[df['regMonth'] != '00']
df['regDate'] = df.apply(lambda x: datetime.datetime.strptime(x['regYear']+'-'+x['regMonth']+'-'+x['regDay'], '%Y-%m-%d'), axis=1)

df['creatYear'] = df['creatDate'].apply(lambda x: x[:4])
df['creatMonth'] = df['creatDate'].apply(lambda x: x[4:6])
df['creatDay'] = df['creatDate'].apply(lambda x: x[6:])
df['creatDate'] = df.apply(lambda x: datetime.datetime.strptime(x['creatYear']+'-'+x['creatMonth']+'-'+x['creatDay'], '%Y-%m-%d'), axis=1)

df['vintage'] = df.apply(lambda x: (x['creatDate']-x['regDate']).days, axis=1)

df['notRepairedDamage'] = pd.Categorical(df['notRepairedDamage'], categories=['-', '0.0', '1.0'], ordered=True).codes

# 降低偏度，使数据尽量呈正态分布
bin_power = [i*10 for i in range(31)]
df["power_bin"] = pd.cut(df["power"], bin_power, right = False, labels = False)
df['power_bin'].fillna(31, inplace=True)
df['power'] = np.log(df['power'] + 1) 

df.drop(['regYear', 'regMonth', 'regDay', 'creatYear', 'creatMonth', 'creatDay', 'regDate', 'creatDate', 'seller', 'offerType'], axis=1, inplace=True)
df.dropna(subset=['model', 'bodyType', 'fuelType', 'gearbox'], axis=0, how='any',inplace=True)

In [5]:
# 计算某个品牌的各种统计数量
groupby_df = df.groupby('brand')
all_info = {}
for item, item_data in groupby_df:
    item_info = {}
    item_data = item_data[item_data['price'] > 0]
    item_info['brand_amount'] = item_data.shape[0]
    item_info['brand_price_max'] = item_data['price'].max()
    item_info['brand_price_min'] = item_data['price'].min()
    item_info['brand_price_avg'] = item_data['price'].mean()
    item_info['brand_price_std'] = item_data['price'].std()
    item_info['brand_price_median'] = item_data['price'].median()
    all_info[item] = item_info
brand_feature = pd.DataFrame(all_info).T.reset_index().rename(columns={'index': 'brand'})
df = pd.merge(df, brand_feature, how='left', on='brand')

In [6]:
def transform_minmax(df: pd.DataFrame, col_name: List):
    """需要标准化转换的dataframe和列名"""
    scaler = MinMaxScaler()
    normalize_data = scaler.fit_transform(df[col_name])
    df[col_name] = normalize_data
    return df


def random_forest_classifier(df: pd.DataFrame, col_name: List):
    """用于缺失值的预测"""
    train = df[df[col_name].notnull().all(axis=1)]
    test = df[df[col_name].isnull().any(axis=1)]
    X_train = train_data.drop(col_name, axis=1)
    y_train = train_data[col_name]
    X_test = test_data.drop(col_name, axis=1)
    
    # 随机森林拟合缺失值
    rf = RandomForestClassifier()
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    df.loc[df[column_list].isnull().any(axis=1), col_name] = y_pred
    return df

# 归一化
df = transform_minmax(df, ["brand_amount", "brand_price_avg", "brand_price_max", "brand_price_median", "brand_price_min", "brand_price_std", "power", "kilometer"])

In [7]:
# LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(df['regionCode'].unique())
label_encoded = label_encoder.transform(df['regionCode'])
df['regionCode'] = label_encoded

In [8]:
# OneHot
# 对类别特征转换为onehot
data = pd.get_dummies(df, columns=['model', 'brand', 'bodyType','fuelType','gearbox', 'notRepairedDamage', 'power_bin'], dummy_na=True)

In [9]:
# 最后删除无关特征
# data = data.drop(['creatDate', "regDate"], axis = 1)

In [10]:
# 扩展数值特征
# from sklearn.preprocessing import PolynomialFeatures

# x = df[['v_0','v_1','v_2', 'v_3', 'v_4', 'v_5','v_6','v_7', 'v_8', 'v_9', 'v_10','v_11','v_12', 'v_13', 'v_14']]
# y = df['price']

# poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)

# x_poly = poly.fit_transform(x)
# poly_df = pd.DataFrame(x_poly, columns=poly.get_feature_names_out())

In [11]:
# data = pd.concat([data, poly_df], axis=1)

In [12]:
corr_data = data.loc[:, ~data.columns.isin(['SaleID', 'name'])]
corr_matrix = corr_data.corr('pearson')
# sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".1f", linewidths=0.1)
# plt.show()

In [13]:
# 训练集测试集区分
train, test = train_test_split(data, test_size=0.3, random_state=42)
X_train = train.loc[:, ~train.columns.isin(['SaleID', 'name', 'price'])]
X_test = test.loc[:, ~test.columns.isin(['SaleID', 'name', 'price'])]
y_train = train['price']
y_test = test['price']
train_id = train['SaleID']
test_id = test['SaleID']

In [14]:
## XGBoost
def random_search_param(train_X, train_y, test_X, test_y):
    fixed_params = {
        'early_stopping_rounds': 300,
        'slient': 1,
        'colsample_bytree': 0.9,
        'eval_metric': 'rmse',
        'objective': 'reg:squarederror'
    }
    params = {
        'n_estimators': [100, 500, 1000],
        'learning_rate': [0.01, 0.1, 0.15, 0.2, 0.3],
        'max_depth': [3, 6, 9, 12],
        'gamma': [0.1, 0.2, 0.5, 1],
        'reg_alpha': [0.1, 0.2, 0.5, 1],
        'reg_lambda': [0.1, 0.5, 1, 5, 10],
        'min_child_weight': [7, 10, 12, 15, 20],
        'subsample': [0.7, 0.8, 0.9, 1],
    }
    model = XGBRegressor(**fixed_params, enable_categorical=True)
    optimized_GBM = RandomizedSearchCV(model, params, cv=5, return_train_score=True, error_score='raise')
    # optimized_GBM = GridSearchCV(model, params, cv=5, return_train_score=True) # 二选一，网格搜索耗时长
    optimized_GBM.fit(train_X, train_y, eval_set=[(test_X, test_y)], verbose=500)
    print('参数的最佳取值：{0}'.format(optimized_GBM.best_params_))
    print('最佳模型得分:{0}'.format(optimized_GBM.best_score_))
    best_params = optimized_GBM.best_params_
    best_params.update(fixed_params)
    return best_params

params = random_search_param(X_train, y_train, X_test, y_test)

[0]	validation_0-rmse:7011.07503
[99]	validation_0-rmse:1432.95502
[0]	validation_0-rmse:7013.61982
[99]	validation_0-rmse:1465.75082
[0]	validation_0-rmse:7018.59128
[99]	validation_0-rmse:1458.41036
[0]	validation_0-rmse:7010.41736
[99]	validation_0-rmse:1446.62918
[0]	validation_0-rmse:7016.01552
[99]	validation_0-rmse:1433.29314
[0]	validation_0-rmse:7615.76760
[500]	validation_0-rmse:1487.95765
[999]	validation_0-rmse:1410.79507
[0]	validation_0-rmse:7616.02435
[500]	validation_0-rmse:1499.61390
[999]	validation_0-rmse:1423.30540
[0]	validation_0-rmse:7616.40160
[500]	validation_0-rmse:1520.68110
[999]	validation_0-rmse:1436.28991
[0]	validation_0-rmse:7615.55965
[500]	validation_0-rmse:1494.54257
[999]	validation_0-rmse:1414.16617
[0]	validation_0-rmse:7615.87417
[500]	validation_0-rmse:1482.35946
[999]	validation_0-rmse:1403.73272
[0]	validation_0-rmse:6317.20948
[499]	validation_0-rmse:1336.23812
[0]	validation_0-rmse:6311.89836
[499]	validation_0-rmse:1350.55168
[0]	validation

In [15]:
xgb_model = XGBRegressor(**params, enable_categorical=True)
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=300
)
y_pred = xgb_model.predict(X_test)
print(mean_absolute_error(y_test, y_pred))

[0]	validation_0-rmse:6667.05702
[300]	validation_0-rmse:1341.35358
[492]	validation_0-rmse:1347.00510
596.727223798414


In [16]:
# LightGB

In [38]:
result = pd.DataFrame(columns=['SaleID', 'price'])
test_x = test_data.loc[:, ~test_data.columns.isin(['SaleID', 'name'])]
xgb_pred = xgb_model.predict(test_x)
result['SaleID'] = test_data['SaleID']
result['price'] = xgb_pred


ValueError: feature_names mismatch: ['model', 'brand', 'power', 'kilometer', 'notRepairedDamage', 'regionCode', 'seller', 'offerType', 'v_0', 'v_1', 'v_2', 'v_3', 'v_4', 'v_5', 'v_6', 'v_7', 'v_8', 'v_9', 'v_10', 'v_11', 'v_12', 'v_13', 'v_14', 'vintage', 'bodyType_0.0', 'bodyType_1.0', 'bodyType_2.0', 'bodyType_3.0', 'bodyType_4.0', 'bodyType_5.0', 'bodyType_6.0', 'bodyType_7.0', 'fuelType_0.0', 'fuelType_1.0', 'fuelType_2.0', 'fuelType_3.0', 'fuelType_4.0', 'fuelType_5.0', 'fuelType_6.0', 'gearbox_0.0', 'gearbox_1.0'] ['model', 'brand', 'power', 'kilometer', 'notRepairedDamage', 'regionCode', 'seller', 'offerType', 'v_0', 'v_1', 'v_2', 'v_3', 'v_4', 'v_5', 'v_6', 'v_7', 'v_8', 'v_9', 'v_10', 'v_11', 'v_12', 'v_13', 'v_14', 'vintage', 'bodyType_0.0', 'bodyType_1.0', 'bodyType_2.0', 'bodyType_3.0', 'bodyType_4.0', 'bodyType_5.0', 'bodyType_6.0', 'bodyType_7.0', 'bodyType_nan', 'fuelType_0.0', 'fuelType_1.0', 'fuelType_2.0', 'fuelType_3.0', 'fuelType_nan', 'fuelType_4.0', 'fuelType_5.0'

In [ ]:
result.to_csv('sample_submit.csv', ignore_index=True)